In [ ]:
# In[1]: Read Documents and Fixed-Size Batching
import csv
import json
import os
import sys

# Increase CSV field size limit for large text fields
csv.field_size_limit(sys.maxsize)

# Read all documents
csv_path = "/home/nena-meijer/PyCharmMiscProject/dataset/VWS_subset/6-VWS_documents_NER_nl_labels.csv"
print(f"Reading CSV from {csv_path}")
docs = []
with open(csv_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for idx, row in enumerate(reader, start=1):
        docs.append({'id': row['document_id'], 'text': row['document_text']})
        if idx % 500 == 0:
            print(f"Loaded {idx} documents")
print(f"Total documents loaded: {len(docs)}")

# Split into fixed-size batches of 100 documents each
BATCH_SIZE = 6
batches = [docs[i:i + BATCH_SIZE] for i in range(0, len(docs), BATCH_SIZE)]
print(f"Created {len(batches)} batches of up to {BATCH_SIZE} docs each.")

# In[2]: Save Batches to Files
output_dir = "batches"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
for idx, batch in enumerate(batches, start=1):
    batch_path = os.path.join(output_dir, f"batch_{idx}.json")
    with open(batch_path, 'w', encoding='utf-8') as bf:
        json.dump({'documents': batch}, bf, ensure_ascii=False, indent=2)
    print(f"Saved batch {idx}/{len(batches)} with {len(batch)} docs to {batch_path}")

In [12]:
API_key="API_KEY"

In [13]:
import os
import json
import re
from google import genai
from google.genai import types

# Setup Gemini client and model
client = genai.Client(api_key=API_key)
MODEL_NAME = "models/gemini-2.0-flash"
print(f"Using model: {MODEL_NAME} for API generation")

# Instruction template
INSTRUCTIONS = (
    "Je krijgt een JSON-array met documenten met betrekking tot de coronacrisis in het Nederlands of Engels.\n"
    "Geef de resultaten terug in het Nederlands. Voor elk document extraheer je de volgende informatie:\n"
    "1. personen: alle namen van personen die **LETTERLIJK** en **VOLLEDIG** genoemd worden in het document en een rol spelen in de coronacrisis of de besluitvorming eromheen. Dit kunnen politici, wetenschappers, artsen, woordvoerders, vertegenwoordigers van organisaties, of andere relevante individuen zijn. Normaliseer de namen niet. Gebruik exact de spelling zoals die in het document staat.\n"
    "De namen moeten minimaal 2 karakters lang zijn.\n"
    "**EXTREEM BELANGRIJK: Genereer ABSOLUUT GEEN namen die NIET LETTERLIJK en VOLLEDIG in het document voorkomen. Er moet een EXACTE overeenkomst zijn. Gebruik GEEN synoniemen, afkortingen, of interpretaties. Indien er geen personen in het document genoemd worden, laat de \"personen\" array dan leeg.**\n"
    "Stuur als output één JSON-object met een \"results\"-array. De \"results\"-array bevat objecten met een \"document_id\" en een \"personen\" array. De \"personen\" array bevat een lijst van strings, waarbij elke string een persoonsnaam is.\n"
    "Neem per document het document_id mee in de response.\n"
    "Lever uitsluitend geldige JSON zonder extra markdown‑fences, zonder trailing commas, met alle strings correct geescaped.\n"
    "Hier zijn een paar voorbeelden:\n"
    "INPUT DOCUMENT 1: 'Vandaag heeft minister H. de Jonge aangekondigd dat de vaccinatiecampagne wordt uitgebreid.'\n"
    "INPUT DOCUMENT 2: 'Volgens professor Van Dissel is dit een cruciale stap. Ook De Jonge is het hiermee eens.'\n"
    "INPUT DOCUMENT 3: 'Hugo heeft vandaag een persconferentie gegeven.'\n"
    "INPUT DOCUMENT 4: 'Dit document beschrijft de algemene situatie rondom corona.'\n"
    "OUTPUT 1: {\"results\": [{\"document_id\": \"voorbeeld-1\", \"personen\": [\"H. de Jonge\"]}]}\n"
    "OUTPUT 2: {\"results\": [{\"document_id\": \"voorbeeld-2\", \"personen\": [\"Van Dissel\", \"De Jonge\"]}]}\n"
    "OUTPUT 3: {\"results\": [{\"document_id\": \"voorbeeld-3\", \"personen\": [\"Hugo\"]}]}\n"
    "OUTPUT 4: {\"results\": [{\"document_id\": \"voorbeeld-4\", \"personen\": []}]}\n"
)

# Batch directory and range
BATCH_DIR = "/home/nena-meijer/PyCharmMiscProject/information_extraction/batches"
BATCH_START = 1471
BATCH_END = 1471  # inclusive

aggregated_results = []

for i in range(BATCH_START, BATCH_END + 1):
    batch_path = os.path.join(BATCH_DIR, f"batch_{i}.json")
    if not os.path.isfile(batch_path):
        print(f"Batch {i}: bestand niet gevonden, overslaan...")
        continue

    with open(batch_path, 'r', encoding='utf-8') as f:
        try:
            batch = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Batch {i}: JSON decode error bij het inlezen van bestand: {e}")
            aggregated_results.append({'batch': i, 'error': f'File load error: {e}', 'raw_response': None})
            continue

    prompt = INSTRUCTIONS + json.dumps({'documents': batch}, ensure_ascii=False, indent=2)
    print(len(prompt))
    print(f"Processing batch {i} with {len(batch)} documents...")

    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(response_mime_type="text/plain")

    try:
        response_text = ''.join(
            chunk.text for chunk in client.models.generate_content_stream(
                model=MODEL_NAME, contents=contents, config=config
            )
        )

        # Clean up potential Markdown code fences
        cleaned = response_text.strip()
        cleaned = re.sub(r"^```json", "", cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
        cleaned = cleaned.strip()

        data = json.loads(cleaned)
        results = data.get('results', [])
        print(f"Batch {i}: parsed {len(results)} results")
        aggregated_results.extend(results)

    except json.JSONDecodeError as e:
        print(f"Batch {i} JSON parse error: {e}\nIncluding raw response in results.json")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': cleaned})

    except Exception as e:
        print(f"Batch {i}: onverwachte fout: {e}")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': None})

# Save final aggregated results
output_path = 'results_persons_1471.json'
with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump({'results': aggregated_results}, outfile, ensure_ascii=False, indent=2)

print(f"Saved aggregated results: {len(aggregated_results)} entries to '{output_path}'")


Using model: models/gemini-2.0-flash for API generation
70072
Processing batch 1471 with 1 documents...
Batch 1471: parsed 6 results
Saved aggregated results: 6 entries to 'results_persons_1471.json'


In [3]:
import json
import csv

# Pad naar je JSON-bestand
json_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/persons/persons_all.json'
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'

# JSON inladen
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Items zonder document_id vinden
valid_results = []
extra_results = []

for item in data['results']:
    if 'document_id' in item:
        valid_results.append(item)
    elif 'raw_response' in item and item['raw_response']:
        # Fix de raw_response van item 2
        raw = item['raw_response']
        try:
            # Corrigeer de extra komma met een simpele vervanging
            raw_fixed = raw.replace(',\n    }', '\n    }')
            parsed_raw = json.loads(raw_fixed)
            extra_results.extend(parsed_raw['results'])
        except Exception as e:
            print("Kon raw_response niet parsen:", e)

# Combineer de correcte results
all_results = valid_results + extra_results

# Schrijf naar CSV
with open(csv_path, 'w', encoding='utf-8', newline='') as f_csv:
    writer = csv.writer(f_csv)
    writer.writerow(['document_id', 'name'])  # header

    for item in all_results:
        document_id = item['document_id']
        personen = item.get('personen', [])
        if personen:
            for persoon in personen:
                writer.writerow([document_id, persoon])
        else:
            writer.writerow([document_id, ''])  # leeg als er geen personen zijn

print(f"Aantal geldige items: {len(all_results)}")
print(f"CSV opgeslagen als: {csv_path}")

Aantal geldige items: 22871
CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/Person.csv


In [5]:
import csv
from collections import Counter

csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'

namen = []

# Lees de CSV en verzamel namen
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:  # alleen niet-lege namen
            namen.append(name)

# Tel de namen
naam_tellingen = Counter(namen)

# Sorteer op count (aflopend)
gesorteerd = naam_tellingen.most_common()

# Print de resultaten
print("Unieke namen en hun aantallen (gesorteerd):")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"Totaal unieke namen: {len(naam_tellingen)}")


Unieke namen en hun aantallen (gesorteerd):
Koen: 458
Hugo: 339
Hugo de Jonge: 234
de Jonge: 129
Rutte: 113
hugo: 104
Halsema: 102
hugo de jonge: 92
de jonge: 85
van Rijn: 84
Asscher: 82
Martin van Rijn: 80
koen: 72
minister de jonge: 70
Marijnissen: 60
Grapperhaus: 58
Jaap van Dissel: 53
De Jonge: 44
martin van rijn: 43
H.M. de Jonge: 42
Martin: 40
H. de Jonge: 38
Ciska: 37
bruno bruins: 35
minister bruins: 34
Wilders: 34
Angelique: 33
Baudet: 33
Hijink: 33
bruno: 33
minister van rijn: 32
Marjolein: 31
Kees: 30
Ploumen: 29
Klaver: 28
Ciska Scheidel: 28
Ellemeet: 27
jonge, h.m. de (hugo): 27
Van Dissel: 27
Van Rijn: 26
Bergkamp: 26
paul blokhuis: 25
Ouwehand: 24
Bruins: 24
van den berg: 24
Jaap: 23
Mark Rutte: 23
Paul Blokhuis: 23
Veldman: 23
bruins: 23
veldman: 22
Bruno Bruins: 21
Diertens: 21
Paul: 21
h.m. de (hugo): 21
angelique: 21
ciska scheidel: 21
Jetten: 20
Blokhuis: 20
mark rutte: 19
Bruno: 19
Krol: 18
Koolmees: 18
Tamara van Ark: 18
Tamara: 18
Aboutaleb: 18
Wiebes: 17
Heerma:

In [12]:
import json
import csv
from collections import Counter

# Pad naar je mapping JSON
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/persons/person_mapping.json'
# Pad naar je CSV
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)  # niet in mapping, origineel laten

# Tel de genormaliseerde namen
naam_tellingen = Counter(namen_genormaliseerd)

# Gesorteerd afdrukken
gesorteerd = naam_tellingen.most_common()

print("Genormaliseerde namen en hun aantallen:")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"\nTotaal unieke genormaliseerde namen: {len(naam_tellingen)}")


Genormaliseerde namen en hun aantallen:
Hugo de Jonge: 899
Koen: 458
Martin van Rijn: 363
Hugo: 339
Bruno Bruins: 207
Mark Rutte: 193
Ciska Scheidel: 130
Femke Halsema: 113
hugo: 104
Lodewijk Asscher: 94
Paul Blokhuis: 88
Ferdinand Grapperhaus: 84
koen: 72
Lilian Marijnissen: 70
Jaap van Dissel: 53
Carola Schouten: 49
Tamara van Ark: 49
Geert Wilders: 45
Jesse Klaver: 39
Lilianne Ploumen: 34
Vera Bergkamp: 34
Angelique: 33
Baudet: 33
Hijink: 33
Marjolein: 31
Wouter Koolmees: 31
Eric Wiebes: 30
Kees: 30
Ellemeet: 27
Van Dissel: 27
Kajsa Ollongren: 25
Ouwehand: 24
van den berg: 24
Jaap: 23
Veldman: 23
veldman: 22
Wopke Hoekstra: 22
Rob Jetten: 21
Diertens: 21
Paul: 21
angelique: 21
Krol: 18
Carla Dik-Faber: 18
Tamara: 18
Aboutaleb: 18
Heerma: 16
Agema: 16
Bill Gates: 16
jaap van dissel: 16
Raymond Knops: 15
Ingrid van Engelshoven: 15
Segers: 14
Ken: 14
Coen: 14
Mona Keijzer: 13
keen: 13
Kees van der Staaij: 13
Keen: 13
Ernst: 13
Kroger: 12
Van der Staaij: 12
Kerstens: 12
Westerveld: 12
R

In [14]:
import json
import csv
from collections import Counter

# Paden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/persons/person_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Person_per_doc.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)

# Tel en haal unieke namen
unieke_namen = sorted(set(namen_genormaliseerd))

# Schrijf unieke namen naar CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.writer(f_out)
    writer.writerow(['person_id', 'name'])  # header
    for idx, naam in enumerate(unieke_namen, start=1):
        writer.writerow([idx, naam])

print(f"Unieke namen opgeslagen in: {csv_output_path}")
print(f"Totaal unieke namen: {len(unieke_namen)}")


Unieke namen opgeslagen in: /home/nena-meijer/PyCharmMiscProject/database/Person.csv
Totaal unieke namen: 4121


In [13]:
import json
import csv

# Paden naar je bestanden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/persons/person_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Person_normalized.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive mapping

# Nieuwe lijst voor rijen met genormaliseerde namen
genormaliseerde_rijen = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                row['name'] = lookup[name_lower]  # vervang met genormaliseerde naam
            else:
                row['name'] = name  # geen match, laat origineel staan
        else:
            row['name'] = ''  # lege waarde blijft leeg
        genormaliseerde_rijen.append(row)

# Schrijf het resultaat naar een nieuwe CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    fieldnames = ['document_id', 'name']
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(genormaliseerde_rijen)

print(f"Genormaliseerde CSV opgeslagen als: {csv_output_path}")


Genormaliseerde CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/Person_normalized.csv


In [16]:
import csv

# Paden naar je CSV's
person_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Person_normalized.csv'
unique_persons_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Person.csv'
output_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/DocumentPerson.csv'

# Stap 1: Laad person_id -> name mapping
person_mapping = {}
with open(unique_persons_csv_path, 'r', encoding='utf-8') as f_unique:
    reader = csv.DictReader(f_unique)
    for row in reader:
        person_mapping[row['name'].strip()] = row['person_id']

# Stap 2: Verwerk de document_id, name CSV
document_person_rows = []

with open(person_csv_path, 'r', encoding='utf-8') as f_persons:
    reader = csv.DictReader(f_persons)
    for row in reader:
        document_id = row['document_id']
        name = row['name'].strip()
        if name and name in person_mapping:
            person_id = person_mapping[name]
            document_person_rows.append({'document_id': document_id, 'person_id': person_id})
        elif name == '':
            # Optioneel: sla lege namen over, of voeg document_id met lege person_id toe
            continue
        else:
            # Naam niet gevonden in mapping, optioneel loggen
            print(f"Naam niet gevonden: {name}")

# Stap 3: Schrijf nieuwe CSV met document_id, person_id
with open(output_csv_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.DictWriter(f_out, fieldnames=['document_id', 'person_id'])
    writer.writeheader()
    writer.writerows(document_person_rows)

print(f"Document-Person CSV opgeslagen als: {output_csv_path}")
print(f"Totaal koppelingen: {len(document_person_rows)}")


Document-Person CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/DocumentPerson.csv
Totaal koppelingen (matches): 9791
Niet gevonden namen: 0
Lege naam rijen: 18426


In [17]:
import csv

# Pad naar je DocumentPerson.csv
document_person_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/DocumentPerson.csv'

# Verzamel alle person_id's
person_ids = set()

with open(document_person_csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        person_ids.add(row['person_id'])

# Resultaat
print(f"Totaal unieke person_id's in DocumentPerson: {len(person_ids)}")


Totaal unieke person_id's in DocumentPerson: 4121
